<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/comfyui_video0425_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Cell 1: 基础环境与图像生成核心 (请最先运行)
# ==========================================
import os
import subprocess

print("=== 🚀 开始安装 ComfyUI 基础与图像模块 ===")
%cd /content

# 1. 拉取 ComfyUI 官方最新源码
if not os.path.exists("/content/ComfyUI/.git"):
    print("⏳ 正在拉取 ComfyUI 官方最新源码...")
    !rm -rf /content/ComfyUI
    !export GIT_TERMINAL_PROMPT=0 && git clone https://github.com/comfyanonymous/ComfyUI
else:
    print("✨ 检测到 ComfyUI 已存在，正在拉取最新核心更新...")
    %cd /content/ComfyUI
    !git pull

%cd /content/ComfyUI

# 2. 安装 ComfyUI 官方运行依赖
print("⏳ 正在安装 ComfyUI 官方运行依赖...")
!pip install -q -r requirements.txt

# 3. 独立安装 huggingface_hub 和 hf_transfer，开启满速下载
print("⏳ 正在配置 Hugging Face 高速下载环境...")
!pip install -q huggingface_hub hf_transfer

# 4. 安装核心图像与管理插件
%cd /content/ComfyUI/custom_nodes

# 【强烈推荐】ComfyUI-Manager (节点管理器)
print("⏳ 正在检查并安装 ComfyUI 节点管理器 (ComfyUI-Manager)...")
if not os.path.exists("ComfyUI-Manager/.git"):
    !export GIT_TERMINAL_PROMPT=0 && git clone https://github.com/ltdrdata/ComfyUI-Manager.git
else:
    !cd ComfyUI-Manager && git pull

# 【基础依赖】rgthree 节点包
print("⏳ 正在检查并安装 rgthree 基础节点包 (rgthree-comfy)...")
if not os.path.exists("rgthree-comfy/.git"):
    !export GIT_TERMINAL_PROMPT=0 && git clone https://github.com/rgthree/rgthree-comfy.git
    !cd rgthree-comfy && if [ -f "requirements.txt" ]; then pip install -q -r requirements.txt; fi
else:
    !cd rgthree-comfy && git pull

%cd /content/ComfyUI
print("\n✅ Cell 1 (基础与图像模块) 安装完成！")

In [ ]:
# ==========================================
# Cell 3: 音频处理与语音克隆模块 (FishAudioS2)
# ==========================================
import os
import subprocess

print("=== 🎵 开始安装音频与语音克隆模块 ===")

# 1. 检查并安装系统级音频底层依赖
print("⏳ 正在检查并安装系统级依赖 (FFmpeg & libsndfile1)...")
!apt update -qq
!apt -y install ffmpeg libsndfile1 -qq

# 2. 强制补充音频处理极易缺失的核心 Python 库
print("⏳ 正在安装音频处理核心拓展库...")
!pip install -q soundfile librosa pydub

%cd /content/ComfyUI/custom_nodes

# 3. 安装 FishAudioS2 多人语音克隆 TTS 插件包
print("⏳ 正在安装 FishAudioS2 语音克隆插件包 (ComfyUI-FishAudioS2)...")
if not os.path.exists("ComfyUI-FishAudioS2/.git"):
    !export GIT_TERMINAL_PROMPT=0 && git clone https://github.com/Saganaki22/ComfyUI-FishAudioS2.git
    !cd ComfyUI-FishAudioS2 && pip install -q -r requirements.txt
else:
    !cd ComfyUI-FishAudioS2 && git pull

%cd /content/ComfyUI
print("\n✅ Cell 3 (音频与语音克隆模块) 安装完成！")

In [ ]:
# ==========================================
# 二、终极模型下载列表 (Flux2 + LTX-2.3 音视频双修顶配版)
# ==========================================
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download

# 1. ⚠️必须开启高速满速下载魔法！
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# 2. 从 Colab Secrets 中读取 HF Token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
except userdata.SecretNotFoundError:
    print("❌ 错误：请先在左侧 '🔑 Secrets' 中添加名为 'HF_TOKEN' 的密钥！")
    raise

# 3. 灵活配置下载任务列表 (共 12 个核心文件)
downloads = [
    # -------------------【第一部分：Flux 图像生态】-------------------
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/diffusion_models/flux2_dev_fp8mixed.safetensors",
        "local_dir": "/content/ComfyUI/models/unet"
    },
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/loras/Flux2TurboComfyv2.safetensors",
        "local_dir": "/content/ComfyUI/models/loras"
    },
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/text_encoders/mistral_3_small_flux2_bf16.safetensors",
        "local_dir": "/content/ComfyUI/models/clip"
    },
    {
        "repo_id": "stronman/flux2-dev",
        "filename": "split_files/vae/flux2-vae.safetensors",
        "local_dir": "/content/ComfyUI/models/vae"
    },
    {
        "repo_id": "Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0",
        "filename": "diffusion_pytorch_model.safetensors",
        "local_dir": "/content/ComfyUI/models/controlnet"
    },
    {
        "repo_id": "xey/sldr_flux_nsfw_v2-studio",
        "filename": "sldr_flux_nsfw_v2-studio.safetensors",
        "local_dir": "/content/ComfyUI/models/loras"
    },

    # -------------------【第二部分：LTX-2.3 音视频生态】-------------------
    # 7. LTX-2.3 视频主模型
   {
        "repo_id": "stronman/LTX-2.3",
        "filename": "ltx-2.3-22b-dev.safetensors",
        "local_dir": "/content/ComfyUI/models/unet"
    },
    # 8. LTX-2.3 钦定 Gemma 3 大脑 (FP8 高清版)
    {
        "repo_id": "Comfy-Org/ltx-2",
        "filename": "split_files/text_encoders/gemma_3_12B_it_fp8_scaled.safetensors",
        "local_dir": "/content/ComfyUI/models/clip"
    },
    # 9. LTX-2.3 文本投影层 (解决维度冲突的关键)
    {
        "repo_id": "stronman/LTX2.3_comfy",
        "filename": "text_encoders/ltx-2.3_text_projection_bf16.safetensors",
        "local_dir": "/content/ComfyUI/models/clip"
    },
    # 10. 通用辅助文本编码器 CLIP-L
    {
        "repo_id": "comfyanonymous/flux_text_encoders",
        "filename": "clip_l.safetensors",
        "local_dir": "/content/ComfyUI/models/clip"
    },
    # 11. 【更新】LTX-2.3 专用视频 VAE (适配 KJNodes)
    {
        "repo_id": "stronman/LTX2.3_comfy",
        "filename": "vae/LTX23_video_vae_bf16.safetensors",
        "local_dir": "/content/ComfyUI/models/vae"
    },
    # 12. 【新增】LTX-2.3 专用音频 VAE (适配 KJNodes)
    {
        "repo_id": "stronman/LTX2.3_comfy",
        "filename": "vae/LTX23_audio_vae_bf16.safetensors",
        "local_dir": "/content/ComfyUI/models/vae"
    }
]

print(f"\n🚀 开始满速同步指定的 {len(downloads)} 个模型文件...\n")

for i, task in enumerate(downloads, 1):
    repo = task["repo_id"]
    file_path = task["filename"]
    target_dir = task["local_dir"]
    file_name = file_path.split('/')[-1]

    print(f"[{i}/{len(downloads)}] 正在从 {repo} 同步: {file_name}")
    try:
        hf_hub_download(
            repo_id=repo,
            filename=file_path,
            local_dir=target_dir,
            repo_type="model"
        )
        print(f"✅ 已完成并存放到: {target_dir}\n")
    except Exception as e:
        print(f"❌ 下载失败 ({file_name}): {e}\n")

print("🎉 恭喜！12 个满血核心组件全部归位，音视频同步生成环境已就绪！")

In [ ]:
# ==========================================
# 三、内网穿透配置 (只需运行一次)
# ==========================================
import os
from google.colab import userdata

# 1. 读取 Colab Secrets
try:
    VPS_IP = userdata.get('VPS_IP')
    FRP_TOKEN = userdata.get('FRP_TOKEN')
except Exception as e:
    print("❌ 错误：无法读取 Secrets！请检查左侧是否填写并开启了 'VPS_IP' 和 'FRP_TOKEN'。")
    raise e

# 2. 下载并解压 FRP 客户端
%cd /content
if not os.path.exists("/content/frp_0.56.0_linux_amd64"):
    print("⏳ 正在下载并解压 FRP 客户端...")
    !wget -q https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz
    !tar -xzf frp_0.56.0_linux_amd64.tar.gz
    !rm frp_0.56.0_linux_amd64.tar.gz

# 3. 动态生成并写入配置文件
print("⏳ 正在生成 frpc.toml 配置文件...")
frpc_config_content = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8080
"""

with open("/content/frp_0.56.0_linux_amd64/frpc.toml", "w", encoding="utf-8") as f:
    f.write(frpc_config_content.strip())

print("✅ FRP 配置文件生成完毕！")

In [ ]:
# ==========================================
# 四、启动 FRP 和 ComfyUI (重启界面时运行),不注册，启动快！访问域名
# ==========================================
import subprocess
import threading
import os
import configparser

import time

# --- 新增：防断开保活机制 ---
def keep_alive():
    while True:
        time.sleep(300)  # 每 5 分钟
        print("\n[Keep-Alive] 保持 Colab 连接活跃中...")

print("⏳ 正在启动防断开后台保活线程...")
threading.Thread(target=keep_alive, daemon=True).start()
# --------------------------

# 1. 在后台新线程启动 FRP
def start_frpc():
    subprocess.run(["/content/frp_0.56.0_linux_amd64/frpc", "-c", "/content/frp_0.56.0_linux_amd64/frpc.toml"])

print("⏳ 正在后台唤起 FRP 穿透服务...")
threading.Thread(target=start_frpc, daemon=True).start()

print("============================================================")
print("✅ FRP 穿透已就绪！")
# 这里已修改为你的指定域名（保留了 8080 端口，如果你的服务端用 Nginx 做了 80 端口反代，可以把 :8080 删掉）
print("👉 启动完成后，请通过浏览器访问: http://cjp.usdream.dpdns.org:8080")
print("============================================================\n")

# ---------------------------------------------------------
# 新增：彻底拦截 ComfyUI-Manager 启动时的联网 Fetch 行为
# ---------------------------------------------------------
print("⏳ 正在配置 Manager 网络模式以跳过 Fetch...")
# 兼容所有版本的配置文件路径 (特别是最新的 __manager 路径)
manager_config_paths = [
    "/content/ComfyUI/user/__manager/config.ini",               # 最新版 V3.39+ 配置路径
    "/content/ComfyUI/user/default/ComfyUI-Manager/config.ini", # 较新版配置路径
    "/content/ComfyUI/custom_nodes/ComfyUI-Manager/config.ini"  # 老旧版配置路径
]

for config_path in manager_config_paths:
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    config = configparser.ConfigParser()

    if os.path.exists(config_path):
        config.read(config_path)

    if 'default' not in config:
        config['default'] = {}

    # 将网络模式改为 private
    config['default']['network_mode'] = 'private'

    with open(config_path, 'w') as f:
        config.write(f)

print("✅ 已成功切断 Fetch ComfyRegistry Data 流程！\n")
# ---------------------------------------------------------

# 2. 启动 ComfyUI 主程序
print("⏳ 正在启动 ComfyUI 主进程...\n")
%cd /content/ComfyUI
!python main.py --dont-print-server